# Exploration of Gene Expression Models

This notebook uses two related gene-expression systems to demonstrate several AutoReduce workflows:

1. **Automatic reduction of a six-state protein-expression model.** AutoReduce enumerates candidate reduced systems, simulates their protein output and ranks them by mean absolute error (MAE).
2. **Structured reduction of an eight-state gene-expression model.** Conservation laws first eliminate bound complexes, after which time-scale separation generates smaller models that retain the protein output.

The goal is not to identify one universally best reduced model. Reduced-model accuracy depends on the selected output, parameter values, initial conditions and time interval, so every candidate should be validated in the regime where it will be used.

## Setup

The notebook requires AutoReduce together with NumPy, SymPy, Matplotlib and Plotly. The automatic search and robustness cells are the most computationally expensive parts of the workflow.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go

from IPython.display import Math, display
from sympy import latex, symbols

from autoreduce import System, load_ODE_model
from autoreduce.utils import get_ODE, get_reducible, get_SSM

## 1. Six-State Protein-Expression Model

The first system tracks free polymerase \(P\), the gene-polymerase complex \(C_1\), transcript \(T\), free ribosome \(R\), the transcript-ribosome complex \(C_2\) and protein \(X\):

\[
x=\begin{bmatrix}P&C_1&T&R&C_2&X\end{bmatrix}^{\mathsf T},
\qquad y=X.
\]

Its dynamics are

\[
\begin{aligned}
\dot P &= (k_{up}+k_{tx})C_1-k_{bp}GP,\\
\dot C_1 &= k_{bp}GP-(k_{up}+k_{tx})C_1,\\
\dot T &= k_{tx}C_1+(k_{ur}+k_{tl})C_2-k_{br}TR-d_TT,\\
\dot R &= (k_{ur}+k_{tl})C_2-k_{br}TR,\\
\dot C_2 &= k_{br}TR-(k_{ur}+k_{tl})C_2,\\
\dot X &= k_{tl}C_2-d_XX.
\end{aligned}
\]

This compact model is useful for demonstrating AutoReduce's automatic candidate search.

### 1.1 Construct the full system

`load_ODE_model` creates symbolic state and parameter vectors. The output matrix `C` selects protein \(X\), the sixth state, as the quantity that reduced models should preserve.

In [ ]:
protein_n = 6
protein_nouts = 1

# Parameter order:
# k_bp, k_up, k_tx, k_br, k_ur, k_tl, d_T, d_X, G
protein_params_values = np.array(
    [10.0, 10.0, 1.0, 10.0, 10.0, 1.0, 0.1, 0.1, 1.0]
)

# State order: P, C1, T, R, C2, X
protein_x_init = np.array([0.0, 100.0, 400.0, 100.0, 0.0, 20.0])

protein_x, protein_f, protein_params = load_ODE_model(
    protein_n,
    len(protein_params_values),
)

protein_f[0] = (
    (protein_params[1] + protein_params[2]) * protein_x[1]
    - protein_params[0] * protein_params[8] * protein_x[0]
)
protein_f[1] = (
    protein_params[0] * protein_params[8] * protein_x[0]
    - (protein_params[1] + protein_params[2]) * protein_x[1]
)
protein_f[2] = (
    protein_params[2] * protein_x[1]
    + (protein_params[4] + protein_params[5]) * protein_x[4]
    - protein_params[3] * protein_x[2] * protein_x[3]
    - protein_params[6] * protein_x[2]
)
protein_f[3] = (
    (protein_params[4] + protein_params[5]) * protein_x[4]
    - protein_params[3] * protein_x[2] * protein_x[3]
)
protein_f[4] = (
    protein_params[3] * protein_x[2] * protein_x[3]
    - (protein_params[4] + protein_params[5]) * protein_x[4]
)
protein_f[5] = (
    protein_params[5] * protein_x[4]
    - protein_params[7] * protein_x[5]
)

protein_C = np.zeros((protein_nouts, protein_n), dtype=int)
protein_C[0, 5] = 1

protein_system = System(
    protein_x,
    protein_f,
    params=protein_params,
    params_values=protein_params_values,
    C=protein_C.tolist(),
    x_init=protein_x_init,
)

### 1.2 Simulate the original model

`get_ODE` converts the symbolic AutoReduce system into a numerical ODE problem. The output is computed as \(y=Cx\).

In [ ]:
protein_timepoints = np.linspace(0.0, 100.0, 200)
protein_full_ode = get_ODE(protein_system, protein_timepoints)
protein_full_solution = protein_full_ode.solve_system().T
protein_full_output = np.ravel(
    np.asarray(protein_system.C) @ protein_full_solution
)

plt.figure(figsize=(8, 4.5))
plt.plot(
    protein_timepoints,
    protein_full_output,
    color="black",
    linewidth=2.5,
)
plt.xlabel("Time")
plt.ylabel("Protein X")
plt.title("Six-State Protein-Expression Model")
plt.tight_layout()
plt.show()

### 1.3 Output sensitivity

AutoReduce's sensitivity-system method (SSM) measures how the selected output responds to each parameter over time. This can help identify influential parameters and interpret why a reduction succeeds or fails.

In [ ]:
protein_ssm_timepoints = np.linspace(0.0, 60.0, 10)
protein_ssm = get_SSM(protein_system, protein_ssm_timepoints)
protein_state_sensitivities = protein_ssm.compute_SSM(normalize=True)

# Dimensions: time x parameter x output
protein_output_sensitivities = np.einsum(
    "os,tps->tpo",
    np.asarray(protein_system.C),
    protein_state_sensitivities,
)

parameter_names = [
    "k_bp", "k_up", "k_tx", "k_br", "k_ur",
    "k_tl", "d_T", "d_X", "G",
]

plt.figure(figsize=(9, 4.5))
plt.imshow(
    protein_output_sensitivities[:, :, 0].T,
    aspect="auto",
    origin="lower",
    cmap="coolwarm",
)
plt.colorbar(label="Normalized sensitivity")
plt.xticks(
    range(len(protein_ssm_timepoints)),
    [f"{time:.1f}" for time in protein_ssm_timepoints],
)
plt.yticks(range(len(parameter_names)), parameter_names)
plt.xlabel("Time")
plt.ylabel("Parameter")
plt.title("Sensitivity of Protein X to Model Parameters")
plt.tight_layout()
plt.show()

### 1.4 Generate candidate reduced models

`get_reducible` adds model-reduction methods to the full system. `reduce_simple()` then searches candidate retained-state sets, evaluates approximation error and computes robustness information. This cell may take several minutes.

In [ ]:
protein_reducer = get_reducible(
    protein_system,
    protein_timepoints,
    protein_ssm_timepoints,
)
protein_reducer.nstates_tol_min = 2
protein_reducer.nstates_tol_max = 5
protein_reducer.error_tol = 100

protein_results = protein_reducer.reduce_simple()
protein_reduced_models = list(protein_results.keys())

print(f"AutoReduce generated {len(protein_reduced_models)} candidate models.")

### 1.5 Compare and rank the candidates

Every candidate is simulated on the same time grid as the full model. The candidates are ranked by the MAE of protein \(X\); the five closest trajectories are shown to keep the figure readable.

In [ ]:
protein_comparisons = []

for model in protein_reduced_models:
    try:
        reduced_ode = get_ODE(model, protein_timepoints)
        reduced_solution = reduced_ode.solve_system().T
        reduced_output = np.ravel(
            np.asarray(model.C) @ reduced_solution
        )
        mae = np.mean(np.abs(protein_full_output - reduced_output))
        protein_comparisons.append(
            {
                "model": model,
                "states": [str(state) for state in model.x],
                "output": reduced_output,
                "mae": float(mae),
            }
        )
    except Exception as error:
        print(f"Skipped states {model.x}: {error}")

protein_comparisons.sort(key=lambda result: result["mae"])

if not protein_comparisons:
    raise RuntimeError("No reduced protein-expression model simulated successfully.")

for rank, result in enumerate(protein_comparisons[:5], start=1):
    print(
        f"{rank}. states={result['states']}, "
        f"MAE={result['mae']:.6g}"
    )

colors = ["#0072B2", "#009E73", "#E69F00", "#CC79A7", "#56B4E9"]
protein_figure = go.Figure()
protein_figure.add_trace(
    go.Scatter(
        x=protein_timepoints,
        y=protein_full_output,
        mode="lines",
        name="Original six-state model",
        line=dict(color="black", dash="dot", width=4),
    )
)

for color, result in zip(colors, protein_comparisons[:5]):
    protein_figure.add_trace(
        go.Scatter(
            x=protein_timepoints,
            y=result["output"],
            mode="lines",
            name=(
                f"States {result['states']} | "
                f"MAE={result['mae']:.3g}"
            ),
            line=dict(color=color, width=2),
        )
    )

protein_figure.update_layout(
    title="Original vs. Reduced Protein Expression Models: Protein X (X)",
    xaxis_title="Time",
    yaxis_title="Protein X",
    legend_title="Five lowest-MAE models",
    template="plotly_white",
    width=1000,
    height=650,
)
protein_figure.show()

# Optional SVG export (requires the Plotly Kaleido package):
# protein_figure.write_image("protein_expression_comparison.svg")

## 2. Eight-State Gene-Expression Model

The extended system adds an enzyme \(E\) and an enzyme-transcript complex \(C_3\). It tracks

\[
x=\begin{bmatrix}P&C_1&T&R&C_2&E&C_3&X\end{bmatrix}^{\mathsf T},
\qquad y=X.
\]

The reaction structure is

\[
\begin{aligned}
G+P &\rightleftharpoons C_1, & C_1 &\rightarrow G+P+T,\\
T+R &\rightleftharpoons C_2, & C_2 &\rightarrow T+R+X,\\
T+E &\rightleftharpoons C_3, & C_3 &\rightarrow E,\\
T &\rightarrow \varnothing, & X &\rightarrow \varnothing.
\end{aligned}
\]

This model demonstrates how known conservation laws can be applied before time-scale separation.

### 2.1 Construct the extended system

The initial free polymerase, ribosome and enzyme concentrations equal their conserved totals. Protein \(X\) is again the selected output.

In [ ]:
(
    p_free,
    c1,
    transcript,
    r_free,
    c2,
    e_free,
    c3,
    protein_x_state,
) = symbols("P C1 T R C2 E C3 X")

gene_states = [
    p_free,
    c1,
    transcript,
    r_free,
    c2,
    e_free,
    c3,
    protein_x_state,
]

(
    k_bp,
    k_up,
    k_tx,
    k_br,
    k_ur,
    k_tl,
    k_be,
    k_ue,
    d_i,
    d_x,
    d_t,
    e_total,
    p_total,
    r_total,
    gene_copy,
) = symbols(
    "k_bp k_up k_tx k_br k_ur k_tl "
    "k_be k_ue d_i d_x d_T E_tot P_tot R_tot G"
)

gene_params = [
    k_bp,
    k_up,
    k_tx,
    k_br,
    k_ur,
    k_tl,
    k_be,
    k_ue,
    d_i,
    d_x,
    d_t,
    e_total,
    p_total,
    r_total,
    gene_copy,
]

gene_params_values = np.array(
    [
        80.0, 2.0, 0.5, 80.0, 2.0,
        0.5, 10.0, 2.0, 0.1, 0.5,
        0.01, 100.0, 100.0, 400.0, 10.0,
    ]
)

gene_f = [
    (k_up + k_tx) * c1 - k_bp * gene_copy * p_free,
    k_bp * gene_copy * p_free - (k_up + k_tx) * c1,
    (
        k_tx * c1
        + (k_ur + k_tl) * c2
        + k_ue * c3
        - k_br * transcript * r_free
        - k_be * transcript * e_free
        - d_t * transcript
    ),
    (k_ur + k_tl) * c2 - k_br * transcript * r_free,
    k_br * transcript * r_free - (k_ur + k_tl) * c2,
    (k_ue + d_i) * c3 - k_be * transcript * e_free,
    k_be * transcript * e_free - (k_ue + d_i) * c3,
    k_tl * c2 - d_x * protein_x_state,
]

gene_x_init = np.zeros(len(gene_states))
gene_x_init[0] = gene_params_values[-3]  # P_tot
gene_x_init[3] = gene_params_values[-2]  # R_tot
gene_x_init[5] = gene_params_values[-4]  # E_tot

gene_C = np.zeros((1, len(gene_states)), dtype=int)
gene_C[0, 7] = 1

gene_system = System(
    gene_states,
    gene_f,
    params=gene_params,
    params_values=gene_params_values,
    C=gene_C.tolist(),
    x_init=gene_x_init,
)

### 2.2 Simulate the original eight-state system

In [ ]:
gene_timepoints = np.linspace(0.0, 24.0, 200)
gene_full_ode = get_ODE(gene_system, gene_timepoints)
gene_full_solution = gene_full_ode.solve_system().T
gene_full_output = np.ravel(
    np.asarray(gene_system.C) @ gene_full_solution
)

plt.figure(figsize=(8, 4.5))
plt.plot(
    gene_timepoints,
    gene_full_output,
    color="black",
    linewidth=2.5,
)
plt.xlabel("Time")
plt.ylabel("Protein X")
plt.title("Eight-State Gene-Expression Model")
plt.tight_layout()
plt.show()

### 2.3 Apply conservation laws

The bound complexes share conserved molecular pools with their free species:

\[
P+C_1=P_{\mathrm{tot}},\qquad
R+C_2=R_{\mathrm{tot}},\qquad
E+C_3=E_{\mathrm{tot}}.
\]

`set_conservation_laws` uses these relationships to eliminate \(C_1\), \(C_2\) and \(C_3\) before any time-scale assumption is introduced.

In [ ]:
gene_ssm_timepoints = np.linspace(0.0, 2.0, 10)
gene_reducer = get_reducible(
    gene_system,
    gene_timepoints,
    gene_ssm_timepoints,
)
gene_reducer.nstates_tol_min = 2
gene_reducer.nstates_tol_max = 3

conserved_quantities = [
    p_free + c1 - p_total,
    r_free + c2 - r_total,
    e_free + c3 - e_total,
]
states_to_eliminate = [c1, c2, c3]

conservation_reduced_rhs = gene_reducer.set_conservation_laws(
    conserved_quantities,
    states_to_eliminate,
)

display(
    Math(
        r"\text{System after conservation-law elimination: }"
        + latex(conservation_reduced_rhs)
    )
)

### 2.4 Apply time-scale separation

Two candidate systems are constructed explicitly:

- \([T,R,X]\), which treats free polymerase and enzyme as fast states.
- \([T,X]\), which additionally treats free ribosome as fast.

AutoReduce solves the fast-state steady-state equations and substitutes those expressions into the retained dynamics.

In [ ]:
gene_reduced_trx, gene_fast_ss_trx = (
    gene_reducer.solve_timescale_separation(
        [transcript, r_free, protein_x_state],
        fast_states=[p_free, e_free],
        debug=False,
    )
)

gene_reduced_tx, gene_fast_ss_tx = (
    gene_reducer.solve_timescale_separation(
        [transcript, protein_x_state],
        fast_states=[p_free, r_free, e_free],
        debug=False,
    )
)

display(
    Math(
        r"\text{Retained states }[T,R,X]: "
        + latex(gene_reduced_trx.f)
    )
)
display(
    Math(
        r"\text{Retained states }[T,X]: "
        + latex(gene_reduced_tx.f)
    )
)

### 2.5 Compare the full and reduced outputs

Both reduced systems are simulated using the same parameters, initial-condition regime and time points as the full model. MAE is reported for protein \(X\).

In [ ]:
gene_candidate_models = [
    ("Retained states [T, R, X]", gene_reduced_trx),
    ("Retained states [T, X]", gene_reduced_tx),
]

gene_comparisons = []
for name, model in gene_candidate_models:
    reduced_ode = get_ODE(model, gene_timepoints)
    reduced_solution = reduced_ode.solve_system().T
    reduced_output = np.ravel(
        np.asarray(model.C) @ reduced_solution
    )
    mae = np.mean(np.abs(gene_full_output - reduced_output))
    gene_comparisons.append(
        {
            "name": name,
            "model": model,
            "output": reduced_output,
            "mae": float(mae),
        }
    )

gene_comparisons.sort(key=lambda result: result["mae"])

gene_figure = go.Figure()
gene_figure.add_trace(
    go.Scatter(
        x=gene_timepoints,
        y=gene_full_output,
        mode="lines",
        name="Original eight-state model",
        line=dict(color="black", dash="dot", width=4),
    )
)

for color, result in zip(["#0072B2", "#D55E00"], gene_comparisons):
    print(f"{result['name']}: MAE={result['mae']:.6g}")
    gene_figure.add_trace(
        go.Scatter(
            x=gene_timepoints,
            y=result["output"],
            mode="lines",
            name=f"{result['name']} | MAE={result['mae']:.3g}",
            line=dict(color=color, width=2.5),
        )
    )

gene_figure.update_layout(
    title="Original vs. Reduced Gene-Expression Models: Protein X (X)",
    xaxis_title="Time",
    yaxis_title="Protein X",
    legend_title="Models ranked by MAE",
    template="plotly_white",
    width=1000,
    height=650,
)
gene_figure.show()

# Optional SVG export (requires the Plotly Kaleido package):
# gene_figure.write_image("gene_expression_comparison.svg")

### 2.6 Evaluate robustness

The robustness metric measures how sensitive the approximation is to parameter perturbations. A low trajectory error at one parameter set is not sufficient if the reduced model is highly fragile to nearby parameter values.

In [ ]:
gene_robustness_trx = gene_reducer.get_robustness_metric(
    gene_reduced_trx
)
gene_robustness_tx = gene_reducer.get_robustness_metric(
    gene_reduced_tx
)

print("[T, R, X] robustness result:")
print(gene_robustness_trx)
print()
print("[T, X] robustness result:")
print(gene_robustness_tx)

## Conclusions

- The six-state example demonstrates automatic candidate generation, numerical validation and error-based ranking.
- The eight-state example demonstrates how domain knowledge can simplify a model through conservation laws before applying time-scale separation.
- Retaining fewer states does not automatically produce the best approximation. Accuracy and robustness depend on the output, parameters, initial conditions and time interval.
- Candidate reduced models should therefore be tested across the situational parameter ranges in which they are intended to operate.